# Home Credit Gold Layer & Risk Reference

This notebook creates the application-level Home Credit Gold dataset from the validated cleaned Silver layer and develops a disciplined traditional-credit risk reference model using Logistic Regression and Random Forest.

Home Credit remains a separate reference environment and is not merged with the Nigerian BNPL population.

In [0]:
# ============================================================
# HOME CREDIT GOLD LAYER & RISK REFERENCE
# Gold Layer Creation and Initial Audit
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1. Paths and reproducibility
# ------------------------------------------------------------

CLEAN_SILVER_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "silver_application_features_clean"
)

GOLD_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "gold_home_credit_application"
)

RANDOM_SEED = 42

# ------------------------------------------------------------
# 2. Load validated cleaned Silver layer
# ------------------------------------------------------------

hc_silver_clean = spark.read.format("delta").load(CLEAN_SILVER_PATH)

# ------------------------------------------------------------
# 3. Core structural validation before Gold creation
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "SK_ID_CURR",
    "TARGET",
    "bureau_total_debt_clean",
    "bureau_max_debt_clean",
    "bureau_debt_to_credit_ratio_clean",
    "bureau_overdue_to_credit_ratio_clean",
    "historical_payment_completion_ratio_clean",
    "historical_overpayment_ratio",
    "historical_overpayment_flag"
]

missing_required_columns = [
    column_name
    for column_name in REQUIRED_COLUMNS
    if column_name not in hc_silver_clean.columns
]

if missing_required_columns:
    raise ValueError(
        f"Required Home Credit columns are missing: "
        f"{missing_required_columns}"
    )

row_count = hc_silver_clean.count()

unique_customer_count = (
    hc_silver_clean
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

duplicate_customer_count = (
    hc_silver_clean
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

target_null_count = (
    hc_silver_clean
    .filter(F.col("TARGET").isNull())
    .count()
)

if row_count != 307511:
    raise ValueError(
        f"Unexpected Home Credit row count: {row_count}. "
        f"Expected 307511."
    )

if unique_customer_count != row_count:
    raise ValueError(
        "SK_ID_CURR is not unique at the application level."
    )

if duplicate_customer_count != 0:
    raise ValueError(
        f"Duplicate SK_ID_CURR groups detected: "
        f"{duplicate_customer_count}"
    )

if target_null_count != 0:
    raise ValueError(
        f"TARGET contains {target_null_count} null observations."
    )

# ------------------------------------------------------------
# 4. Create application-level Gold dataset
# ------------------------------------------------------------
# Gold retains the validated application-level analytical table.
# No BNPL records are introduced and no target transformation is
# performed.

hc_gold = hc_silver_clean

# ------------------------------------------------------------
# 5. Persist Gold layer
# ------------------------------------------------------------

(
    hc_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PATH)
)

# Reload persisted Gold layer to verify the physical output
hc_gold_verified = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

# ------------------------------------------------------------
# 6. Gold-layer validation
# ------------------------------------------------------------

gold_row_count = hc_gold_verified.count()

gold_unique_customer_count = (
    hc_gold_verified
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

gold_duplicate_customer_count = (
    hc_gold_verified
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

gold_column_count = len(hc_gold_verified.columns)

if gold_row_count != row_count:
    raise ValueError(
        "Gold row count does not match the validated Silver layer."
    )

if gold_unique_customer_count != gold_row_count:
    raise ValueError(
        "Gold layer does not contain one row per SK_ID_CURR."
    )

if gold_duplicate_customer_count != 0:
    raise ValueError(
        "Duplicate SK_ID_CURR values detected in Gold layer."
    )

# ------------------------------------------------------------
# 7. Target distribution
# ------------------------------------------------------------

target_distribution = (
    hc_gold_verified
    .groupBy("TARGET")
    .count()
    .withColumn(
        "share",
        F.col("count") / F.lit(gold_row_count)
    )
    .orderBy("TARGET")
)

# ------------------------------------------------------------
# 8. Schema inventory
# ------------------------------------------------------------

numeric_types = (
    T.ByteType,
    T.ShortType,
    T.IntegerType,
    T.LongType,
    T.FloatType,
    T.DoubleType,
    T.DecimalType
)

numeric_columns = [
    field.name
    for field in hc_gold_verified.schema.fields
    if isinstance(field.dataType, numeric_types)
]

categorical_columns = [
    field.name
    for field in hc_gold_verified.schema.fields
    if isinstance(field.dataType, T.StringType)
]

date_columns = [
    field.name
    for field in hc_gold_verified.schema.fields
    if isinstance(field.dataType, (T.DateType, T.TimestampType))
]

# ------------------------------------------------------------
# 9. Missingness audit
# ------------------------------------------------------------

missingness_summary = (
    hc_gold_verified
    .select([
        F.sum(
            F.when(F.col(column_name).isNull(), 1).otherwise(0)
        ).alias(column_name)
        for column_name in hc_gold_verified.columns
    ])
    .collect()[0]
    .asDict()
)

missingness_df = (
    spark.createDataFrame(
        [
            (
                column_name,
                int(missing_count),
                float(missing_count / gold_row_count)
            )
            for column_name, missing_count in missingness_summary.items()
        ],
        ["column_name", "missing_count", "missing_rate"]
    )
    .orderBy(F.desc("missing_rate"))
)

# ------------------------------------------------------------
# 10. Identify modelling-sensitive columns
# ------------------------------------------------------------

identifier_columns = [
    column_name
    for column_name in hc_gold_verified.columns
    if column_name.upper().startswith("SK_ID_")
]

target_columns = ["TARGET"]

date_like_columns = [
    column_name
    for column_name in hc_gold_verified.columns
    if (
        "DAYS_" in column_name.upper()
        or "DATE" in column_name.upper()
        or "MONTHS_BALANCE" in column_name.upper()
    )
]

# ------------------------------------------------------------
# 11. Concise audit output
# ------------------------------------------------------------

print("=" * 70)
print("HOME CREDIT GOLD LAYER AUDIT")
print("=" * 70)

print(f"Gold path:                  {GOLD_PATH}")
print(f"Rows:                       {gold_row_count:,}")
print(f"Unique SK_ID_CURR:          {gold_unique_customer_count:,}")
print(f"Duplicate customer groups: {gold_duplicate_customer_count:,}")
print(f"Columns:                    {gold_column_count:,}")
print(f"Numeric columns:            {len(numeric_columns):,}")
print(f"Categorical columns:        {len(categorical_columns):,}")
print(f"Date columns:               {len(date_columns):,}")
print(f"Identifier columns:         {len(identifier_columns):,}")
print(f"Date-like columns:          {len(date_like_columns):,}")
print(f"TARGET nulls:               {target_null_count:,}")

print("\nTARGET DISTRIBUTION")
target_distribution.show(truncate=False)

print("TOP MISSINGNESS")
missingness_df.filter(
    F.col("missing_count") > 0
).limit(20).show(truncate=False)

print("IDENTIFIER COLUMNS")
print(identifier_columns)

print("\nDATE-LIKE / TEMPORAL COLUMNS")
print(date_like_columns)

print("\nGOLD LAYER VALIDATION: PASSED")

HOME CREDIT GOLD LAYER AUDIT
Gold path:                  /Volumes/workspace/default/home_credit_raw/gold_home_credit_application
Rows:                       307,511
Unique SK_ID_CURR:          307,511
Duplicate customer groups: 0
Columns:                    180
Numeric columns:            164
Categorical columns:        16
Date columns:               0
Identifier columns:         1
Date-like columns:          6
TARGET nulls:               0

TARGET DISTRIBUTION
+------+------+-------------------+
|TARGET|count |share              |
+------+------+-------------------+
|0     |282686|0.9192711805431351 |
|1     |24825 |0.08072881945686496|
+------+------+-------------------+

TOP MISSINGNESS
+----------------------------+-------------+------------------+
|column_name                 |missing_count|missing_rate      |
+----------------------------+-------------+------------------+
|historical_cc_dpd_def_months|229577       |0.7465651635226057|
|historical_max_cc_balance   |229577       |0

## Model Feature Preparation

The Home Credit reference models use application-level and aggregated historical credit information from the validated Gold layer. The target remains the original application-level `TARGET`.

Identifier fields and raw temporal fields are excluded from the modelling matrix. Historical and behavioural aggregates engineered from bureau, previous applications, installments, POS cash and credit-card histories are retained where available.

In [0]:
# Load the persisted Gold dataset
hc_gold = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)

# Columns that must not enter the modelling matrix
EXCLUDED_COLUMNS = {
    "SK_ID_CURR",
    "TARGET",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
    "bureau_max_days_overdue"
}

# Candidate feature columns
candidate_columns = [
    column_name
    for column_name in hc_gold.columns
    if column_name not in EXCLUDED_COLUMNS
]

# Separate numerical and categorical modelling features
feature_schema = {
    field.name: field.dataType
    for field in hc_gold.schema.fields
}

numeric_feature_columns = [
    column_name
    for column_name in candidate_columns
    if isinstance(feature_schema[column_name], numeric_types)
]

categorical_feature_columns = [
    column_name
    for column_name in candidate_columns
    if isinstance(feature_schema[column_name], T.StringType)
]

# Remove TARGET if it was included through any schema-based selection
numeric_feature_columns = [
    column_name
    for column_name in numeric_feature_columns
    if column_name != "TARGET"
]

categorical_feature_columns = [
    column_name
    for column_name in categorical_feature_columns
    if column_name != "TARGET"
]

# Inspect cardinality of categorical variables before encoding
categorical_cardinality = (
    hc_gold
    .select([
        F.approx_count_distinct(F.col(column_name)).alias(column_name)
        for column_name in categorical_feature_columns
    ])
    .collect()[0]
    .asDict()
)

categorical_cardinality_df = (
    spark.createDataFrame(
        [
            (column_name, int(cardinality))
            for column_name, cardinality in categorical_cardinality.items()
        ],
        ["feature", "distinct_values"]
    )
    .orderBy(F.desc("distinct_values"))
)

# Check whether any feature has effectively no variation
numeric_variation = (
    hc_gold
    .select([
        F.countDistinct(F.col(column_name)).alias(column_name)
        for column_name in numeric_feature_columns
    ])
    .collect()[0]
    .asDict()
)

constant_numeric_features = [
    column_name
    for column_name, distinct_count in numeric_variation.items()
    if distinct_count <= 1
]

# Remove constant numerical features
numeric_feature_columns = [
    column_name
    for column_name in numeric_feature_columns
    if column_name not in constant_numeric_features
]

# Final modelling feature inventory
model_feature_columns = (
    numeric_feature_columns +
    categorical_feature_columns
)

if not model_feature_columns:
    raise ValueError("No modelling features remain after feature screening.")

if "TARGET" in model_feature_columns:
    raise ValueError("TARGET leakage detected in modelling feature set.")

if "SK_ID_CURR" in model_feature_columns:
    raise ValueError("Identifier leakage detected in modelling feature set.")

# Create the modelling dataframe
hc_model_base = (
    hc_gold
    .select(
        "SK_ID_CURR",
        "TARGET",
        *model_feature_columns
    )
)

# Validate feature matrix structure
model_row_count = hc_model_base.count()
model_column_count = len(hc_model_base.columns)

if model_row_count != gold_row_count:
    raise ValueError(
        "Model base row count does not match the Gold dataset."
    )

if (
    hc_model_base
    .select("SK_ID_CURR")
    .distinct()
    .count()
    != model_row_count
):
    raise ValueError(
        "SK_ID_CURR uniqueness was lost during feature preparation."
    )

print("=" * 70)
print("HOME CREDIT MODEL FEATURE INVENTORY")
print("=" * 70)

print(f"Rows:                       {model_row_count:,}")
print(f"Total model features:      {len(model_feature_columns):,}")
print(f"Numerical features:        {len(numeric_feature_columns):,}")
print(f"Categorical features:       {len(categorical_feature_columns):,}")
print(f"Constant features removed: {len(constant_numeric_features):,}")

print("\nCATEGORICAL FEATURE CARDINALITY")
categorical_cardinality_df.show(truncate=False)

print("\nNUMERICAL FEATURES")
print(numeric_feature_columns)

print("\nCATEGORICAL FEATURES")
print(categorical_feature_columns)

print("\nMODEL BASE PREPARATION: PASSED")

HOME CREDIT MODEL FEATURE INVENTORY
Rows:                       307,511
Total model features:      172
Numerical features:        156
Categorical features:       16
Constant features removed: 0

CATEGORICAL FEATURE CARDINALITY
+--------------------------+---------------+
|feature                   |distinct_values|
+--------------------------+---------------+
|ORGANIZATION_TYPE         |56             |
|OCCUPATION_TYPE           |17             |
|NAME_INCOME_TYPE          |8              |
|NAME_TYPE_SUITE           |7              |
|WEEKDAY_APPR_PROCESS_START|7              |
|WALLSMATERIAL_MODE        |7              |
|NAME_HOUSING_TYPE         |6              |
|NAME_EDUCATION_TYPE       |5              |
|NAME_FAMILY_STATUS        |5              |
|FONDKAPREMONT_MODE        |4              |
|CODE_GENDER               |3              |
|HOUSETYPE_MODE            |3              |
|NAME_CONTRACT_TYPE        |2              |
|FLAG_OWN_CAR              |2              |
|FLAG_OW

## Reference Model Training

Two supervised reference models are trained on the Home Credit application-level target: Logistic Regression and Random Forest.

Both models use the same feature set, preprocessing logic, train-validation population and class-weighting approach so that their performance comparison is methodologically consistent.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier
)

# ============================================================
# Model Training and Validation
# ============================================================

RANDOM_SEED = 42

# ------------------------------------------------------------
# Train-validation split
# ------------------------------------------------------------

train_df, validation_df = hc_model_base.randomSplit(
    [0.80, 0.20],
    seed=RANDOM_SEED
)

train_count = train_df.count()
validation_count = validation_df.count()

if train_count == 0 or validation_count == 0:
    raise ValueError(
        "Train-validation split produced an empty dataset."
    )

# ------------------------------------------------------------
# Validate target distribution
# ------------------------------------------------------------

train_target_summary = (
    train_df
    .groupBy("TARGET")
    .count()
    .orderBy("TARGET")
)

validation_target_summary = (
    validation_df
    .groupBy("TARGET")
    .count()
    .orderBy("TARGET")
)

train_target_rate = (
    train_df
    .agg(F.avg("TARGET").alias("target_rate"))
    .first()["target_rate"]
)

validation_target_rate = (
    validation_df
    .agg(F.avg("TARGET").alias("target_rate"))
    .first()["target_rate"]
)

# ------------------------------------------------------------
# Calculate class weights using training data only
# ------------------------------------------------------------

negative_count = (
    train_df
    .filter(F.col("TARGET") == 0)
    .count()
)

positive_count = (
    train_df
    .filter(F.col("TARGET") == 1)
    .count()
)

if negative_count == 0 or positive_count == 0:
    raise ValueError(
        "Both TARGET classes must be present in the training dataset."
    )

total_train = negative_count + positive_count

negative_weight = (
    total_train / (2.0 * negative_count)
)

positive_weight = (
    total_train / (2.0 * positive_count)
)

train_df = train_df.withColumn(
    "class_weight",
    F.when(
        F.col("TARGET") == 1,
        F.lit(positive_weight)
    ).otherwise(
        F.lit(negative_weight)
    )
)

# ------------------------------------------------------------
# Numerical preprocessing
# ------------------------------------------------------------

numeric_input_columns = [
    column_name
    for column_name in numeric_feature_columns
    if column_name in hc_model_base.columns
    and column_name not in {"SK_ID_CURR", "TARGET"}
]

numeric_imputed_columns = [
    f"{column_name}_imputed"
    for column_name in numeric_input_columns
]

numeric_imputer = Imputer(
    inputCols=numeric_input_columns,
    outputCols=numeric_imputed_columns,
    strategy="median"
)

# Vector for Logistic Regression
lr_numeric_assembler = VectorAssembler(
    inputCols=numeric_imputed_columns,
    outputCol="lr_numeric_features",
    handleInvalid="keep"
)

lr_scaler = StandardScaler(
    inputCol="lr_numeric_features",
    outputCol="lr_numeric_scaled",
    withMean=False,
    withStd=True
)

# Vector for Random Forest
rf_numeric_assembler = VectorAssembler(
    inputCols=numeric_imputed_columns,
    outputCol="rf_numeric_features",
    handleInvalid="keep"
)

# ------------------------------------------------------------
# Categorical preprocessing
# ------------------------------------------------------------

categorical_input_columns = [
    column_name
    for column_name in categorical_feature_columns
    if column_name in hc_model_base.columns
    and column_name not in {"SK_ID_CURR", "TARGET"}
]

categorical_index_columns = [
    f"{column_name}_indexed"
    for column_name in categorical_input_columns
]

categorical_encoded_columns = [
    f"{column_name}_encoded"
    for column_name in categorical_input_columns
]

categorical_indexers = [
    StringIndexer(
        inputCol=column_name,
        outputCol=indexed_column,
        handleInvalid="keep"
    )
    for column_name, indexed_column
    in zip(
        categorical_input_columns,
        categorical_index_columns
    )
]

categorical_encoder = OneHotEncoder(
    inputCols=categorical_index_columns,
    outputCols=categorical_encoded_columns,
    handleInvalid="keep"
)

categorical_assembler = VectorAssembler(
    inputCols=categorical_encoded_columns,
    outputCol="categorical_features",
    handleInvalid="keep"
)

# ------------------------------------------------------------
# Final feature vectors
# ------------------------------------------------------------

lr_feature_assembler = VectorAssembler(
    inputCols=[
        "lr_numeric_scaled",
        "categorical_features"
    ],
    outputCol="features",
    handleInvalid="keep"
)

rf_feature_assembler = VectorAssembler(
    inputCols=[
        "rf_numeric_features",
        "categorical_features"
    ],
    outputCol="features",
    handleInvalid="keep"
)

# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="TARGET",
    weightCol="class_weight",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    rawPredictionCol="lr_raw_prediction",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(
    stages=[
        numeric_imputer,
        lr_numeric_assembler,
        lr_scaler,
        *categorical_indexers,
        categorical_encoder,
        categorical_assembler,
        lr_feature_assembler,
        logistic_regression
    ]
)

# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="TARGET",
    weightCol="class_weight",
    predictionCol="rf_prediction",
    probabilityCol="rf_probability",
    rawPredictionCol="rf_raw_prediction",
    numTrees=200,
    maxDepth=12,
    minInstancesPerNode=20,
    featureSubsetStrategy="sqrt",
    seed=RANDOM_SEED
)

rf_pipeline = Pipeline(
    stages=[
        numeric_imputer,
        rf_numeric_assembler,
        *categorical_indexers,
        categorical_encoder,
        categorical_assembler,
        rf_feature_assembler,
        random_forest
    ]
)

# ------------------------------------------------------------
# Train Logistic Regression
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING HOME CREDIT REFERENCE MODELS")
print("=" * 70)

print("\nTraining Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)

print("Logistic Regression training completed.")

# ------------------------------------------------------------
# Train Random Forest
# ------------------------------------------------------------

print("\nTraining Random Forest...")
rf_model = rf_pipeline.fit(train_df)

print("Random Forest training completed.")

TRAINING HOME CREDIT REFERENCE MODELS

Training Logistic Regression...
Logistic Regression training completed.

Training Random Forest...
Random Forest training completed.


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5146087400197454>, line 330
    317 rf_validation = (
    318     rf_model
    319     .transform(validation_df)
   (...)
    326     )
    327 )
    329 # Cache predictions because they will be reused extensively
--> 330 lr_validation.cache()
    331 rf_validation.cache()
    333 lr_validation_count = lr_validation.count()

File <command-5146087400197454>, line 306
    299 print("Random Forest training completed.")
    301 # ------------------------------------------------------------
    302 # Generate validation predictions
    303 # ------------------------------------------------------------
    305 lr_validation = (
--> 306     lr_model
    307     .transform(validation_df)
    308     .select(
    309         "SK_ID_CURR",
    310         "TARGET",
    311         "lr_prediction",
    312         "lr_probability",
 

In [0]:
# ------------------------------------------------------------
# Generate validation predictions
# ------------------------------------------------------------

lr_validation = (
    lr_model
    .transform(validation_df)
    .select(
        "SK_ID_CURR",
        "TARGET",
        "lr_prediction",
        "lr_probability",
        "lr_raw_prediction"
    )
)

rf_validation = (
    rf_model
    .transform(validation_df)
    .select(
        "SK_ID_CURR",
        "TARGET",
        "rf_prediction",
        "rf_probability",
        "rf_raw_prediction"
    )
)

lr_validation_count = lr_validation.count()
rf_validation_count = rf_validation.count()

if lr_validation_count != validation_count:
    raise ValueError(
        "Logistic Regression prediction count does not match "
        "the validation dataset."
    )

if rf_validation_count != validation_count:
    raise ValueError(
        "Random Forest prediction count does not match "
        "the validation dataset."
    )

lr_probability_nulls = (
    lr_validation
    .filter(F.col("lr_probability").isNull())
    .count()
)

rf_probability_nulls = (
    rf_validation
    .filter(F.col("rf_probability").isNull())
    .count()
)

if lr_probability_nulls > 0:
    raise ValueError(
        f"Logistic Regression produced {lr_probability_nulls} "
        "null probability predictions."
    )

if rf_probability_nulls > 0:
    raise ValueError(
        f"Random Forest produced {rf_probability_nulls} "
        "null probability predictions."
    )

print("=" * 70)
print("VALIDATION PREDICTIONS")
print("=" * 70)

print(f"Validation observations: {validation_count:,}")
print(f"LR predictions:           {lr_validation_count:,}")
print(f"RF predictions:           {rf_validation_count:,}")

print("\nValidation prediction generation: PASSED")

VALIDATION PREDICTIONS
Validation observations: 61,343
LR predictions:           61,343
RF predictions:           61,343

Validation prediction generation: PASSED


## Model Evaluation

The reference models are evaluated on the same validation population using ROC-AUC, PR-AUC, precision, recall, F1, KS and Gini. The evaluation uses the original application-level TARGET without class weighting so that reported performance reflects the observed Home Credit population.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# ------------------------------------------------------------
# Convert Spark probability vectors to scalar default scores
# ------------------------------------------------------------

lr_eval = (
    lr_validation
    .withColumn(
        "score",
        vector_to_array("lr_probability")[1]
    )
    .select(
        "SK_ID_CURR",
        "TARGET",
        "score",
        "lr_prediction"
    )
)

rf_eval = (
    rf_validation
    .withColumn(
        "score",
        vector_to_array("rf_probability")[1]
    )
    .select(
        "SK_ID_CURR",
        "TARGET",
        "score",
        "rf_prediction"
    )
)

# ------------------------------------------------------------
# Validate probability extraction
# ------------------------------------------------------------

lr_null_scores = (
    lr_eval
    .filter(F.col("score").isNull())
    .count()
)

rf_null_scores = (
    rf_eval
    .filter(F.col("score").isNull())
    .count()
)

if lr_null_scores > 0:
    raise ValueError(
        f"Logistic Regression produced {lr_null_scores} null scores."
    )

if rf_null_scores > 0:
    raise ValueError(
        f"Random Forest produced {rf_null_scores} null scores."
    )

# ------------------------------------------------------------
# ROC-AUC
# ------------------------------------------------------------

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="TARGET",
    rawPredictionCol="score",
    metricName="areaUnderROC"
)

lr_roc_auc = roc_evaluator.evaluate(lr_eval)
rf_roc_auc = roc_evaluator.evaluate(rf_eval)

# ------------------------------------------------------------
# PR-AUC
# ------------------------------------------------------------

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="TARGET",
    rawPredictionCol="score",
    metricName="areaUnderPR"
)

lr_pr_auc = pr_evaluator.evaluate(lr_eval)
rf_pr_auc = pr_evaluator.evaluate(rf_eval)

# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

def classification_metrics(df, prediction_column):

    metrics = (
        df
        .select(
            F.col("TARGET").cast("double").alias("actual"),
            F.col(prediction_column).cast("double").alias("prediction")
        )
        .agg(
            F.sum(
                F.when(
                    (F.col("actual") == 1) &
                    (F.col("prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("tp"),

            F.sum(
                F.when(
                    (F.col("actual") == 0) &
                    (F.col("prediction") == 1),
                    1
                ).otherwise(0)
            ).alias("fp"),

            F.sum(
                F.when(
                    (F.col("actual") == 0) &
                    (F.col("prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("tn"),

            F.sum(
                F.when(
                    (F.col("actual") == 1) &
                    (F.col("prediction") == 0),
                    1
                ).otherwise(0)
            ).alias("fn")
        )
        .first()
    )

    tp = int(metrics["tp"])
    fp = int(metrics["fp"])
    tn = int(metrics["tn"])
    fn = int(metrics["fn"])

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    accuracy = (
        (tp + tn) / (tp + tn + fp + fn)
        if (tp + tn + fp + fn) > 0
        else 0.0
    )

    return {
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1": f1,
        "Accuracy": accuracy
    }


lr_metrics = classification_metrics(
    lr_eval,
    "lr_prediction"
)

rf_metrics = classification_metrics(
    rf_eval,
    "rf_prediction"
)

# ------------------------------------------------------------
# KS statistic
# ------------------------------------------------------------

def calculate_ks(df):

    class_counts = (
        df
        .groupBy("TARGET")
        .count()
        .collect()
    )

    class_count_dict = {
        int(row["TARGET"]): int(row["count"])
        for row in class_counts
    }

    total_positive = class_count_dict.get(1, 0)
    total_negative = class_count_dict.get(0, 0)

    if total_positive == 0 or total_negative == 0:
        raise ValueError(
            "Both target classes are required for KS calculation."
        )

    score_window = (
        Window
        .orderBy(F.col("score").desc())
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )

    ks_df = (
        df
        .withColumn(
            "cum_positive",
            F.sum(
                F.when(
                    F.col("TARGET") == 1,
                    1
                ).otherwise(0)
            ).over(score_window)
        )
        .withColumn(
            "cum_negative",
            F.sum(
                F.when(
                    F.col("TARGET") == 0,
                    1
                ).otherwise(0)
            ).over(score_window)
        )
        .withColumn(
            "cum_positive_rate",
            F.col("cum_positive") /
            F.lit(total_positive)
        )
        .withColumn(
            "cum_negative_rate",
            F.col("cum_negative") /
            F.lit(total_negative)
        )
        .withColumn(
            "ks_difference",
            F.abs(
                F.col("cum_positive_rate") -
                F.col("cum_negative_rate")
            )
        )
    )

    return (
        ks_df
        .agg(
            F.max("ks_difference").alias("ks")
        )
        .first()["ks"]
    )


lr_ks = calculate_ks(lr_eval)
rf_ks = calculate_ks(rf_eval)

# ------------------------------------------------------------
# Gini
# ------------------------------------------------------------

lr_gini = (2 * lr_roc_auc) - 1
rf_gini = (2 * rf_roc_auc) - 1

# ------------------------------------------------------------
# Validation prevalence
# ------------------------------------------------------------

validation_default_rate = (
    lr_eval
    .agg(
        F.avg("TARGET").alias("default_rate")
    )
    .first()["default_rate"]
)

# ------------------------------------------------------------
# Model comparison
# ------------------------------------------------------------

comparison_rows = [
    (
        "Logistic Regression",
        float(lr_roc_auc),
        float(lr_pr_auc),
        float(lr_ks),
        float(lr_gini),
        float(lr_metrics["Precision"]),
        float(lr_metrics["Recall"]),
        float(lr_metrics["F1"]),
        float(lr_metrics["Specificity"]),
        float(lr_metrics["Accuracy"])
    ),
    (
        "Random Forest",
        float(rf_roc_auc),
        float(rf_pr_auc),
        float(rf_ks),
        float(rf_gini),
        float(rf_metrics["Precision"]),
        float(rf_metrics["Recall"]),
        float(rf_metrics["F1"]),
        float(rf_metrics["Specificity"]),
        float(rf_metrics["Accuracy"])
    )
]

comparison_schema = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "KS",
    "Gini",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "Accuracy"
]

model_comparison = spark.createDataFrame(
    comparison_rows,
    comparison_schema
)

# ------------------------------------------------------------
# Confusion matrix summary
# ------------------------------------------------------------

confusion_rows = [
    (
        "Logistic Regression",
        lr_metrics["TP"],
        lr_metrics["FP"],
        lr_metrics["TN"],
        lr_metrics["FN"]
    ),
    (
        "Random Forest",
        rf_metrics["TP"],
        rf_metrics["FP"],
        rf_metrics["TN"],
        rf_metrics["FN"]
    )
]

confusion_matrix_summary = spark.createDataFrame(
    confusion_rows,
    ["Model", "TP", "FP", "TN", "FN"]
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 70)
print("HOME CREDIT REFERENCE MODEL EVALUATION")
print("=" * 70)

print(f"\nValidation observations: {validation_count:,}")
print(f"Validation default rate: {validation_default_rate:.4%}")

print("\nMODEL PERFORMANCE")
model_comparison.show(truncate=False)

print("CONFUSION MATRICES")
confusion_matrix_summary.show(truncate=False)

print("\nLOGISTIC REGRESSION")
print(f"ROC-AUC:     {lr_roc_auc:.6f}")
print(f"PR-AUC:      {lr_pr_auc:.6f}")
print(f"KS:          {lr_ks:.6f}")
print(f"Gini:        {lr_gini:.6f}")
print(f"Precision:   {lr_metrics['Precision']:.6f}")
print(f"Recall:      {lr_metrics['Recall']:.6f}")
print(f"F1:          {lr_metrics['F1']:.6f}")
print(f"Specificity: {lr_metrics['Specificity']:.6f}")
print(f"Accuracy:    {lr_metrics['Accuracy']:.6f}")

print("\nRANDOM FOREST")
print(f"ROC-AUC:     {rf_roc_auc:.6f}")
print(f"PR-AUC:      {rf_pr_auc:.6f}")
print(f"KS:          {rf_ks:.6f}")
print(f"Gini:        {rf_gini:.6f}")
print(f"Precision:   {rf_metrics['Precision']:.6f}")
print(f"Recall:      {rf_metrics['Recall']:.6f}")
print(f"F1:          {rf_metrics['F1']:.6f}")
print(f"Specificity: {rf_metrics['Specificity']:.6f}")
print(f"Accuracy:    {rf_metrics['Accuracy']:.6f}")

print("\nMODEL EVALUATION: PASSED")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


HOME CREDIT REFERENCE MODEL EVALUATION

Validation observations: 61,343
Validation default rate: 7.9765%

MODEL PERFORMANCE
+-------------------+------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+
|Model              |ROC_AUC           |PR_AUC             |KS                 |Gini              |Precision          |Recall            |F1                 |Specificity       |Accuracy          |
+-------------------+------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+
|Logistic Regression|0.7580760624575837|0.2347625335989643 |0.3889727249046332 |0.5161521249151675|0.16506953658656445|0.6864909053750256|0.26614372870612474|0.6990256864481842|0.6980258546207391|
|Random Forest      |0.7436785907526468|0.22449325081903077|0.36234129955901284|0.487357

## Feature Importance

Feature importance is used to identify which application and historical-credit variables contribute most strongly to the Home Credit reference models.

For Logistic Regression, standardized model coefficients are used as the measure of directional feature contribution. For Random Forest, impurity-based feature importance is used to identify the most influential model inputs.

These results are interpreted as predictive contribution rather than causal effects.

In [0]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ------------------------------------------------------------
# Build exact source-feature labels for the numerical section
# ------------------------------------------------------------

# Numerical features appear first in both final feature vectors.
# Their order is identical to numeric_input_columns.

numeric_feature_count = len(numeric_input_columns)

# ------------------------------------------------------------
# Obtain categorical feature metadata from the fitted LR model
# ------------------------------------------------------------

lr_one_row = (
    lr_model
    .transform(validation_df.limit(1))
)

lr_feature_metadata = (
    lr_one_row
    .schema["features"]
    .metadata
)

lr_attrs = lr_feature_metadata.get(
    "ml_attr",
    {}
).get(
    "attrs",
    {}
)

lr_all_attributes = []

for group in lr_attrs.values():
    lr_all_attributes.extend(group)

lr_all_attributes = sorted(
    lr_all_attributes,
    key=lambda x: x["idx"]
)

lr_feature_count = len(lr_all_attributes)

lr_classifier = lr_model.stages[-1]

lr_coefficients = np.array(
    lr_classifier.coefficients.toArray()
)

if len(lr_coefficients) != lr_feature_count:
    raise ValueError(
        f"LR coefficient count ({len(lr_coefficients)}) does not "
        f"match feature metadata count ({lr_feature_count})."
    )

# ------------------------------------------------------------
# Build reliable LR feature labels
# ------------------------------------------------------------

lr_feature_labels = []

for index in range(lr_feature_count):

    if index < numeric_feature_count:
        label = numeric_input_columns[index]

    else:
        attribute = lr_all_attributes[index]

        label = attribute.get(
            "name",
            f"encoded_feature_{index}"
        )

    lr_feature_labels.append(label)

# ------------------------------------------------------------
# Logistic Regression importance
# ------------------------------------------------------------

lr_importance_rows = [
    (
        int(index),
        lr_feature_labels[index],
        float(lr_coefficients[index]),
        float(abs(lr_coefficients[index]))
    )
    for index in range(len(lr_coefficients))
]

lr_importance_df = (
    spark.createDataFrame(
        lr_importance_rows,
        [
            "feature_index",
            "feature",
            "coefficient",
            "absolute_coefficient"
        ]
    )
    .orderBy(
        F.desc("absolute_coefficient")
    )
)

# ------------------------------------------------------------
# Obtain actual RF feature metadata
# ------------------------------------------------------------

rf_one_row = (
    rf_model
    .transform(validation_df.limit(1))
)

rf_feature_metadata = (
    rf_one_row
    .schema["features"]
    .metadata
)

rf_attrs = rf_feature_metadata.get(
    "ml_attr",
    {}
).get(
    "attrs",
    {}
)

rf_all_attributes = []

for group in rf_attrs.values():
    rf_all_attributes.extend(group)

rf_all_attributes = sorted(
    rf_all_attributes,
    key=lambda x: x["idx"]
)

rf_feature_count = len(rf_all_attributes)

rf_classifier = rf_model.stages[-1]

rf_importances = np.array(
    rf_classifier.featureImportances.toArray()
)

if len(rf_importances) != rf_feature_count:
    raise ValueError(
        f"RF importance count ({len(rf_importances)}) does not "
        f"match feature metadata count ({rf_feature_count})."
    )

# ------------------------------------------------------------
# Build reliable RF feature labels
# ------------------------------------------------------------

rf_feature_labels = []

for index in range(rf_feature_count):

    if index < numeric_feature_count:
        label = numeric_input_columns[index]

    else:
        attribute = rf_all_attributes[index]

        label = attribute.get(
            "name",
            f"encoded_feature_{index}"
        )

    rf_feature_labels.append(label)

# ------------------------------------------------------------
# Random Forest importance
# ------------------------------------------------------------

rf_importance_rows = [
    (
        int(index),
        rf_feature_labels[index],
        float(rf_importances[index])
    )
    for index in range(len(rf_importances))
]

rf_importance_df = (
    spark.createDataFrame(
        rf_importance_rows,
        [
            "feature_index",
            "feature",
            "importance"
        ]
    )
    .orderBy(
        F.desc("importance")
    )
)

# ------------------------------------------------------------
# Display top numerical and categorical features separately
# ------------------------------------------------------------

print("=" * 70)
print("TOP LOGISTIC REGRESSION FEATURES")
print("=" * 70)

lr_importance_df.limit(20).show(
    truncate=False
)

print("=" * 70)
print("TOP RANDOM FOREST FEATURES")
print("=" * 70)

rf_importance_df.limit(20).show(
    truncate=False
)

# ------------------------------------------------------------
# Source-level RF importance
# ------------------------------------------------------------

def clean_feature_name(feature_name):

    if feature_name in numeric_input_columns:
        return feature_name

    for column_name in categorical_input_columns:
        if feature_name.startswith(
            f"categorical_features_{column_name}"
        ):
            return column_name

        if feature_name.startswith(
            f"{column_name}_"
        ):
            return column_name

    return feature_name


clean_feature_udf = F.udf(
    clean_feature_name,
    T.StringType()
)

rf_source_importance = (
    rf_importance_df
    .withColumn(
        "source_feature",
        clean_feature_udf("feature")
    )
    .groupBy("source_feature")
    .agg(
        F.sum("importance").alias(
            "total_importance"
        )
    )
    .orderBy(
        F.desc("total_importance")
    )
)

# ------------------------------------------------------------
# Source-level LR importance
# ------------------------------------------------------------

lr_source_importance = (
    lr_importance_df
    .withColumn(
        "source_feature",
        clean_feature_udf("feature")
    )
    .groupBy("source_feature")
    .agg(
        F.sum("absolute_coefficient").alias(
            "total_absolute_coefficient"
        ),
        F.sum("coefficient").alias(
            "net_coefficient"
        )
    )
    .orderBy(
        F.desc("total_absolute_coefficient")
    )
)

# ------------------------------------------------------------
# Feature-family classification
# ------------------------------------------------------------

def feature_family(column_name):

    name = column_name.lower()

    if name.startswith("bureau_"):
        return "Bureau history"

    if name.startswith("previous_"):
        return "Previous applications"

    if name.startswith("historical_"):
        return "Repayment / account history"

    if name.startswith("ext_source"):
        return "External credit sources"

    if name.startswith("flag_document"):
        return "Documentation"

    if name.startswith("amt_"):
        return "Application amounts"

    if name in categorical_input_columns:
        return "Categorical application profile"

    if name in {
        "cnt_children",
        "cnt_fam_members",
        "region_population_relative",
        "region_rating_client",
        "region_rating_client_w_city"
    }:
        return "Customer / regional profile"

    return "Application / other"


feature_family_udf = F.udf(
    feature_family,
    T.StringType()
)

rf_family_summary = (
    rf_source_importance
    .withColumn(
        "feature_family",
        feature_family_udf("source_feature")
    )
    .groupBy("feature_family")
    .agg(
        F.sum("total_importance").alias(
            "total_importance"
        )
    )
    .orderBy(
        F.desc("total_importance")
    )
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("=" * 70)
print("TOP RANDOM FOREST SOURCE FEATURES")
print("=" * 70)

rf_source_importance.limit(20).show(
    truncate=False
)

print("=" * 70)
print("TOP LOGISTIC REGRESSION SOURCE FEATURES")
print("=" * 70)

lr_source_importance.limit(20).show(
    truncate=False
)

print("=" * 70)
print("RANDOM FOREST FEATURE-FAMILY CONTRIBUTION")
print("=" * 70)

rf_family_summary.show(
    truncate=False
)

print("=" * 70)
print("FEATURE IMPORTANCE VALIDATION")
print("=" * 70)

print(f"LR coefficient count: {len(lr_coefficients):,}")
print(f"LR feature metadata:  {lr_feature_count:,}")
print(f"RF importance count:  {len(rf_importances):,}")
print(f"RF feature metadata:  {rf_feature_count:,}")

if (
    len(lr_coefficients) != lr_feature_count
    or
    len(rf_importances) != rf_feature_count
):
    raise ValueError(
        "Feature importance dimensions do not match model vectors."
    )

print("\nFEATURE IMPORTANCE EXTRACTION: PASSED")

TOP LOGISTIC REGRESSION FEATURES
+-------------+------------------------------------------------------------------+--------------------+--------------------+
|feature_index|feature                                                           |coefficient         |absolute_coefficient|
+-------------+------------------------------------------------------------------+--------------------+--------------------+
|182          |categorical_features_NAME_INCOME_TYPE_encoded_Student             |-3.9687422860630064 |3.9687422860630064  |
|161          |categorical_features_CODE_GENDER_encoded_XNA                      |-3.5126044334020827 |3.5126044334020827  |
|197          |categorical_features_NAME_FAMILY_STATUS_encoded_Unknown           |-2.4201082956136877 |2.4201082956136877  |
|183          |categorical_features_NAME_INCOME_TYPE_encoded_Businessman         |-2.2678994607901206 |2.2678994607901206  |
|190          |categorical_features_NAME_EDUCATION_TYPE_encoded_Academic degree  |-1.0966468

## Final Home Credit Reference Model

Logistic Regression is selected as the primary Home Credit risk-reference model because it provides stronger overall ranking and discrimination performance across ROC-AUC, PR-AUC, KS, Gini and recall.

Random Forest is retained as the nonlinear benchmark. Although it achieves slightly higher precision and F1 at the default classification threshold, Logistic Regression provides the stronger overall discriminatory profile for the reference-model objective.

The selected Home Credit model remains a separate traditional-credit reference engine and is not used as a BNPL default model.

In [0]:
# Final reference-model selection and output persistence

# ------------------------------------------------------------
# Model selection
# ------------------------------------------------------------

FINAL_REFERENCE_MODEL = "Logistic Regression"

final_model_metrics = {
    "Model": FINAL_REFERENCE_MODEL,
    "ROC_AUC": float(lr_roc_auc),
    "PR_AUC": float(lr_pr_auc),
    "KS": float(lr_ks),
    "Gini": float(lr_gini),
    "Precision": float(lr_metrics["Precision"]),
    "Recall": float(lr_metrics["Recall"]),
    "F1": float(lr_metrics["F1"]),
    "Specificity": float(lr_metrics["Specificity"]),
    "Accuracy": float(lr_metrics["Accuracy"])
}

# ------------------------------------------------------------
# Final reference risk scores
# ------------------------------------------------------------

final_reference_predictions = (
    lr_eval
    .select(
        "SK_ID_CURR",
        "TARGET",
        F.col("score").alias("reference_risk_score"),
        F.col("lr_prediction").alias("reference_prediction")
    )
    .withColumn(
        "reference_model",
        F.lit(FINAL_REFERENCE_MODEL)
    )
)

# ------------------------------------------------------------
# Risk-score distribution
# ------------------------------------------------------------

score_summary = (
    final_reference_predictions
    .agg(
        F.count("*").alias("observations"),
        F.min("reference_risk_score").alias("min_score"),
        F.expr(
            "percentile_approx(reference_risk_score, 0.25)"
        ).alias("q25_score"),
        F.expr(
            "percentile_approx(reference_risk_score, 0.50)"
        ).alias("median_score"),
        F.expr(
            "percentile_approx(reference_risk_score, 0.75)"
        ).alias("q75_score"),
        F.max("reference_risk_score").alias("max_score"),
        F.avg("reference_risk_score").alias("mean_score")
    )
)

# ------------------------------------------------------------
# Risk bands for reference-model interpretation
# ------------------------------------------------------------

final_reference_predictions = (
    final_reference_predictions
    .withColumn(
        "risk_band",
        F.when(
            F.col("reference_risk_score") < 0.10,
            F.lit("Very Low")
        )
        .when(
            F.col("reference_risk_score") < 0.20,
            F.lit("Low")
        )
        .when(
            F.col("reference_risk_score") < 0.35,
            F.lit("Moderate")
        )
        .when(
            F.col("reference_risk_score") < 0.50,
            F.lit("High")
        )
        .otherwise(
            F.lit("Very High")
        )
    )
)

risk_band_summary = (
    final_reference_predictions
    .groupBy("risk_band")
    .agg(
        F.count("*").alias("customers"),
        F.avg("reference_risk_score").alias("avg_risk_score"),
        F.avg("TARGET").alias("observed_default_rate")
    )
    .withColumn(
        "customer_share",
        F.col("customers") /
        F.lit(validation_count)
    )
    .orderBy(
        F.when(
            F.col("risk_band") == "Very Low", 1
        )
        .when(
            F.col("risk_band") == "Low", 2
        )
        .when(
            F.col("risk_band") == "Moderate", 3
        )
        .when(
            F.col("risk_band") == "High", 4
        )
        .otherwise(5)
    )
)

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

FINAL_REFERENCE_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "home_credit_risk_reference_predictions"
)

MODEL_COMPARISON_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "home_credit_reference_model_comparison"
)

LR_IMPORTANCE_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "home_credit_lr_feature_importance"
)

RF_IMPORTANCE_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "home_credit_rf_feature_importance"
)

# ------------------------------------------------------------
# Persist final reference outputs
# ------------------------------------------------------------

(
    final_reference_predictions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FINAL_REFERENCE_PATH)
)

(
    model_comparison
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(MODEL_COMPARISON_PATH)
)

(
    lr_source_importance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(LR_IMPORTANCE_PATH)
)

(
    rf_source_importance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(RF_IMPORTANCE_PATH)
)

# ------------------------------------------------------------
# Reload final reference output for verification
# ------------------------------------------------------------

final_reference_verified = (
    spark.read
    .format("delta")
    .load(FINAL_REFERENCE_PATH)
)

final_reference_count = (
    final_reference_verified.count()
)

final_reference_unique = (
    final_reference_verified
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

final_reference_target_nulls = (
    final_reference_verified
    .filter(F.col("TARGET").isNull())
    .count()
)

final_reference_score_nulls = (
    final_reference_verified
    .filter(
        F.col("reference_risk_score").isNull()
    )
    .count()
)

# ------------------------------------------------------------
# Quality checks
# ------------------------------------------------------------

if final_reference_count != validation_count:
    raise ValueError(
        "Final reference output row count does not match "
        "the validation population."
    )

if final_reference_unique != validation_count:
    raise ValueError(
        "Final reference output does not contain one row "
        "per validation application."
    )

if final_reference_target_nulls != 0:
    raise ValueError(
        "TARGET contains null values in the final reference output."
    )

if final_reference_score_nulls != 0:
    raise ValueError(
        "Reference risk scores contain null values."
    )

if FINAL_REFERENCE_MODEL != "Logistic Regression":
    raise ValueError(
        "Unexpected final reference model selection."
    )

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 70)
print("FINAL HOME CREDIT REFERENCE MODEL")
print("=" * 70)

print(f"Selected model: {FINAL_REFERENCE_MODEL}")

print("\nFINAL MODEL METRICS")

for metric_name, metric_value in final_model_metrics.items():
    if metric_name != "Model":
        print(f"{metric_name:<15}: {metric_value:.6f}")

print("\nREFERENCE SCORE DISTRIBUTION")
score_summary.show(truncate=False)

print("REFERENCE RISK BANDS")
risk_band_summary.show(truncate=False)

print("\nOUTPUT PATHS")
print(f"Reference predictions: {FINAL_REFERENCE_PATH}")
print(f"Model comparison:      {MODEL_COMPARISON_PATH}")
print(f"LR importance:         {LR_IMPORTANCE_PATH}")
print(f"RF importance:         {RF_IMPORTANCE_PATH}")

print("\n" + "=" * 70)
print("HOME CREDIT REFERENCE MODEL QUALITY GATE")
print("=" * 70)

print(f"Validation rows:       {final_reference_count:,}")
print(f"Unique applications:   {final_reference_unique:,}")
print(f"TARGET nulls:          {final_reference_target_nulls:,}")
print(f"Score nulls:           {final_reference_score_nulls:,}")
print(f"Selected model:        {FINAL_REFERENCE_MODEL}")

print("\nFINAL REFERENCE MODEL QUALITY GATE: PASSED")

FINAL HOME CREDIT REFERENCE MODEL
Selected model: Logistic Regression

FINAL MODEL METRICS
ROC_AUC        : 0.758076
PR_AUC         : 0.234763
KS             : 0.388973
Gini           : 0.516152
Precision      : 0.165070
Recall         : 0.686491
F1             : 0.266144
Specificity    : 0.699026
Accuracy       : 0.698026

REFERENCE SCORE DISTRIBUTION
+------------+----------------------+------------------+------------------+------------------+-----------------+------------------+
|observations|min_score             |q25_score         |median_score      |q75_score         |max_score        |mean_score        |
+------------+----------------------+------------------+------------------+------------------+-----------------+------------------+
|61343       |1.5543122344752192E-15|0.2563242300191908|0.3926431358191461|0.5621877137948177|0.991147122453752|0.4180043347153951|
+------------+----------------------+------------------+------------------+------------------+-----------------+-----

## Final Home Credit Reference Interpretation

The Home Credit reference modelling exercise demonstrates how traditional application-level and historical-credit information can be processed and evaluated using distributed Spark workflows.

Logistic Regression is retained as the primary reference model, achieving a ROC-AUC of approximately 0.758, PR-AUC of 0.235, KS of 0.389 and Gini of 0.516 on the validation population. Random Forest provides a nonlinear benchmark with slightly lower overall discrimination but higher precision and F1 at the evaluated classification threshold.

Feature analysis shows that the reference models make substantial use of external credit-source variables, bureau history, repayment behaviour, previous applications and application-level financial characteristics. In particular, the Random Forest assigns strong importance to the external credit-source variables and historical credit and repayment features.

The resulting risk scores demonstrate clear ordering across the reference risk bands, with observed default rates increasing from the Very Low band to the Very High band. These scores are used for risk ranking and reference analysis rather than treated as calibrated probabilities of default.

The Home Credit model remains a separate traditional-credit reference engine. Its results are not merged with, substituted for, or treated as a competing model against the Nigerian BNPL population. The purpose of this component is to demonstrate multi-table credit-history engineering and provide a disciplined reference framework for comparison with the information structure of the BNPL modelling environment.

All results are based on the Home Credit dataset and should be interpreted within its population and data limitations.